<a href="https://colab.research.google.com/github/Leonavarro2287/Modelos2026/blob/main/DEA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
# @title CCR - Orientación Input/Output
!pip install pulp openpyxl -q

import pandas as pd
import numpy as np
from pulp import *
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io

from IPython.display import HTML, display

def preparar_datos(df_original):
    vars_fila = df_original.iloc[:, 0].values
    dmus = df_original.columns[1:].values
    data = {v: [] for v in vars_fila}
    for dmu in dmus:
        for v in vars_fila:
            val = df_original[df_original.iloc[:, 0] == v][dmu].values[0]
            data[v].append(val)
    df_proc = pd.DataFrame(data, index=dmus)
    df_proc.index.name = 'DMU'
    df_proc.reset_index(inplace=True)
    return df_proc, vars_fila, dmus

def resolver_ccr(df, col_dmu, inputs, outputs, orientacion):
    n = len(df)
    dmu_names = df[col_dmu].values
    X = df[inputs].values.T
    Y = df[outputs].values.T
    m, s = len(inputs), len(outputs)
    resultados = {'eficiencia': {}, 'referentes': {}, 'metas_inputs': {}, 'metas_outputs': {}}
    EPS = 1e-6

    for k in range(n):
        if orientacion == "Input":
            prob = LpProblem(f"CCR_Input_{k}", LpMinimize)
            theta = LpVariable("theta", lowBound=0, upBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            # Minimizar theta + EPS*(suma holguras)
            prob += theta + EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == theta * X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == Y[r,k]
            # Sin restricción de convexidad (CCR)
            prob.solve(PULP_CBC_CMD(msg=0))
            theta_opt = value(theta)
            eficiencia = theta_opt
            lambdas_opt = [value(l) for l in lambdas]
            holg_in = [value(s) for s in s_menos]
            holg_out = [value(s) for s in s_plus]
            metas_input = {inputs[i]: theta_opt * X[i,k] - holg_in[i] for i in range(m)}
            metas_output = {outputs[r]: Y[r,k] + holg_out[r] for r in range(s)}
        else:  # Output
            prob = LpProblem(f"CCR_Output_{k}", LpMaximize)
            phi = LpVariable("phi", lowBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            # Maximizar phi - EPS*(suma holguras)
            prob += phi - EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == phi * Y[r,k]
            prob.solve(PULP_CBC_CMD(msg=0))
            phi_opt = value(phi)
            eficiencia = 1.0 / phi_opt if phi_opt > 0 else 0
            lambdas_opt = [value(l) for l in lambdas]
            holg_in = [value(s) for s in s_menos]
            holg_out = [value(s) for s in s_plus]
            metas_input = {inputs[i]: X[i,k] - holg_in[i] for i in range(m)}
            metas_output = {outputs[r]: phi_opt * Y[r,k] + holg_out[r] for r in range(s)}

        resultados['eficiencia'][dmu_names[k]] = eficiencia
        resultados['referentes'][dmu_names[k]] = {
            dmu_names[j]: lambdas_opt[j] for j in range(n) if lambdas_opt[j] > 1e-6
        }
        resultados['metas_inputs'][dmu_names[k]] = metas_input
        resultados['metas_outputs'][dmu_names[k]] = metas_output
    return resultados

def mostrar_tabla(resultados, inputs, outputs, dmus, orientacion):
    if orientacion == "Input":
        rows = ['θ*', 'λ*'] + [f"{inp}*" for inp in inputs]
    else:
        rows = ['θ*', 'λ*'] + [f"{out}*" for out in outputs]
    data = {row: [] for row in rows}
    for dmu in dmus:
        eff = resultados['eficiencia'][dmu]
        data['θ*'].append(f"{eff:.4f}".replace('.', ','))
        if eff < 0.9999:
            refs = resultados['referentes'][dmu]
            if refs:
                # Ordenar por peso descendente y crear líneas separadas con <br>
                items = sorted(refs.items(), key=lambda x: x[1], reverse=True)
                texto_html = "<br>".join([f"{r} ({w:.2f})".replace('.', ',') for r, w in items])
                data['λ*'].append(texto_html)
            else:
                data['λ*'].append("")
        else:
            data['λ*'].append("")
        if orientacion == "Input":
            if eff < 0.9999:
                metas = resultados['metas_inputs'][dmu]
                for inp in inputs:
                    val = metas[inp]
                    data[f"{inp}*"].append(f"{val:.2f}".replace('.', ','))
            else:
                for inp in inputs:
                    data[f"{inp}*"].append("")
        else:
            if eff < 0.9999:
                metas = resultados['metas_outputs'][dmu]
                for out in outputs:
                    val = metas[out]
                    data[f"{out}*"].append(f"{val:.2f}".replace('.', ','))
            else:
                for out in outputs:
                    data[f"{out}*"].append("")
    df_resultado = pd.DataFrame(data, index=dmus).T
    # Convertir a HTML y permitir contenido HTML (para los <br>)
    html = df_resultado.to_html(escape=False, na_rep='')
    # Centrar texto y permitir saltos de línea
    html = html.replace('<td>', '<td style="text-align:center; vertical-align:middle;">')
    display(HTML(html))

print("📂 Sube archivo Excel (variables en filas, DMUs en columnas)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_orig = pd.read_excel(io.BytesIO(uploaded[filename]))
df_proc, variables, dmus = preparar_datos(df_orig)
print("Vista previa:")
display(df_orig)

inputs_sel = widgets.SelectMultiple(options=list(variables), description='Inputs:')
outputs_sel = widgets.SelectMultiple(options=list(variables), description='Outputs:')
orientacion_sel = widgets.RadioButtons(options=['Input', 'Output'], description='Orientación:', value='Input')
display(inputs_sel, outputs_sel, orientacion_sel)

btn = widgets.Button(description="Resolver CCR")
out = widgets.Output()
def on_click(b):
    with out:
        clear_output()
        try:
            inputs = list(inputs_sel.value)
            outputs = list(outputs_sel.value)
            orientacion = orientacion_sel.value
            if not inputs or not outputs:
                print("Selecciona inputs y outputs")
                return
            resultados = resolver_ccr(df_proc, 'DMU', inputs, outputs, orientacion)
            print(f"\nMODELO CCR (CRS) - Orientación {orientacion}\n")
            mostrar_tabla(resultados, inputs, outputs, dmus, orientacion)
        except Exception as e:
            print(f"Error: {e}")
btn.on_click(on_click)
display(btn, out)

📂 Sube archivo Excel (variables en filas, DMUs en columnas)


Saving DEA Centros Medicos.xlsx to DEA Centros Medicos (9).xlsx
Vista previa:


,Unnamed: 0,Villa Cabrera,Cerro,Alta Cordoba,Alberdi,Jardin,Poeta Lugones
0,nº de medicos,5,12,8,4,15,7
1,Presupuesto (millones de pesos),20,60,35,15,80,25
2,nº de pacientes atendidos por dia,100,350,210,90,450,180
3,nº de cirugias por dia,10,45,25,8,60,20


SelectMultiple(description='Inputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de paci…

SelectMultiple(description='Outputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de pac…

RadioButtons(description='Orientación:', options=('Input', 'Output'), value='Input')

Button(description='Resolver CCR', style=ButtonStyle())

Output()

In [ ]:
# @title BCC - Orientación Input/Output
!pip install pulp openpyxl -q

import pandas as pd
import numpy as np
from pulp import *
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io

def preparar_datos(df_original):
    vars_fila = df_original.iloc[:, 0].values
    dmus = df_original.columns[1:].values
    data = {v: [] for v in vars_fila}
    for dmu in dmus:
        for v in vars_fila:
            val = df_original[df_original.iloc[:, 0] == v][dmu].values[0]
            data[v].append(val)
    df_proc = pd.DataFrame(data, index=dmus)
    df_proc.index.name = 'DMU'
    df_proc.reset_index(inplace=True)
    return df_proc, vars_fila, dmus

def resolver_bcc(df, col_dmu, inputs, outputs, orientacion):
    n = len(df)
    dmu_names = df[col_dmu].values
    X = df[inputs].values.T
    Y = df[outputs].values.T
    m, s = len(inputs), len(outputs)
    resultados = {'eficiencia': {}, 'referentes': {}, 'metas_inputs': {}, 'metas_outputs': {}}
    EPS = 1e-6

    for k in range(n):
        if orientacion == "Input":
            prob = LpProblem(f"BCC_Input_{k}", LpMinimize)
            theta = LpVariable("theta", lowBound=0, upBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            prob += theta + EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == theta * X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == Y[r,k]
            prob += lpSum(lambdas) == 1
            prob.solve(PULP_CBC_CMD(msg=0))
            theta_opt = value(theta)
            eficiencia = theta_opt
            lambdas_opt = [value(l) for l in lambdas]
            holg_in = [value(s) for s in s_menos]
            holg_out = [value(s) for s in s_plus]
            metas_input = {inputs[i]: theta_opt * X[i,k] - holg_in[i] for i in range(m)}
            metas_output = {outputs[r]: Y[r,k] + holg_out[r] for r in range(s)}
        else:  # Output
            prob = LpProblem(f"BCC_Output_{k}", LpMaximize)
            phi = LpVariable("phi", lowBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            prob += phi - EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == phi * Y[r,k]
            prob += lpSum(lambdas) == 1
            prob.solve(PULP_CBC_CMD(msg=0))
            phi_opt = value(phi)
            eficiencia = 1.0 / phi_opt if phi_opt > 0 else 0
            lambdas_opt = [value(l) for l in lambdas]
            holg_in = [value(s) for s in s_menos]
            holg_out = [value(s) for s in s_plus]
            metas_input = {inputs[i]: X[i,k] - holg_in[i] for i in range(m)}
            metas_output = {outputs[r]: phi_opt * Y[r,k] + holg_out[r] for r in range(s)}

        resultados['eficiencia'][dmu_names[k]] = eficiencia
        resultados['referentes'][dmu_names[k]] = {
            dmu_names[j]: lambdas_opt[j] for j in range(n) if lambdas_opt[j] > 1e-6
        }
        resultados['metas_inputs'][dmu_names[k]] = metas_input
        resultados['metas_outputs'][dmu_names[k]] = metas_output
    return resultados

def mostrar_tabla(resultados, inputs, outputs, dmus, orientacion):
    if orientacion == "Input":
        rows = ['θ*', 'λ*'] + [f"{inp}*" for inp in inputs]
    else:
        rows = ['θ*', 'λ*'] + [f"{out}*" for out in outputs]
    data = {row: [] for row in rows}
    for dmu in dmus:
        eff = resultados['eficiencia'][dmu]
        data['θ*'].append(f"{eff:.4f}".replace('.', ','))
        if eff < 0.9999:
            refs = resultados['referentes'][dmu]
            if refs:
                # Ordenar por peso descendente y crear líneas separadas con <br>
                items = sorted(refs.items(), key=lambda x: x[1], reverse=True)
                # Construir HTML con saltos de línea
                texto_html = "<br>".join([f"{r} ({w:.2f})".replace('.', ',') for r, w in items])
                data['λ*'].append(texto_html)
            else:
                data['λ*'].append("")
        else:
            data['λ*'].append("")
        if orientacion == "Input":
            if eff < 0.9999:
                metas = resultados['metas_inputs'][dmu]
                for inp in inputs:
                    val = metas[inp]
                    data[f"{inp}*"].append(f"{val:.2f}".replace('.', ','))
            else:
                for inp in inputs:
                    data[f"{inp}*"].append("")
        else:
            if eff < 0.9999:
                metas = resultados['metas_outputs'][dmu]
                for out in outputs:
                    val = metas[out]
                    data[f"{out}*"].append(f"{val:.2f}".replace('.', ','))
            else:
                for out in outputs:
                    data[f"{out}*"].append("")
    df_resultado = pd.DataFrame(data, index=dmus).T
    # Convertir a HTML y permitir contenido HTML (para los <br>)
    html = df_resultado.to_html(escape=False, na_rep='')
    # Centrar texto y permitir saltos de línea
    html = html.replace('<td>', '<td style="text-align:center; vertical-align:middle;">')
    display(HTML(html))

print("📂 Sube archivo Excel (variables en filas, DMUs en columnas)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_orig = pd.read_excel(io.BytesIO(uploaded[filename]))
df_proc, variables, dmus = preparar_datos(df_orig)
print("Vista previa:")
display(df_orig)

inputs_sel = widgets.SelectMultiple(options=list(variables), description='Inputs:')
outputs_sel = widgets.SelectMultiple(options=list(variables), description='Outputs:')
orientacion_sel = widgets.RadioButtons(options=['Input', 'Output'], description='Orientación:', value='Input')
display(inputs_sel, outputs_sel, orientacion_sel)

btn = widgets.Button(description="Resolver BCC (sin holguras, λ vertical)")
out = widgets.Output()
def on_click(b):
    with out:
        clear_output()
        try:
            inputs = list(inputs_sel.value)
            outputs = list(outputs_sel.value)
            orientacion = orientacion_sel.value
            if not inputs or not outputs:
                print("Selecciona inputs y outputs")
                return
            resultados = resolver_bcc(df_proc, 'DMU', inputs, outputs, orientacion)
            print(f"\nMODELO BCC (VRS) - Orientación {orientacion}\n")
            mostrar_tabla(resultados, inputs, outputs, dmus, orientacion)
        except Exception as e:
            print(f"Error: {e}")
btn.on_click(on_click)
display(btn, out)

📂 Sube archivo Excel (variables en filas, DMUs en columnas)


Saving DEA Centros Medicos.xlsx to DEA Centros Medicos (2).xlsx
Vista previa:


,Unnamed: 0,Villa Cabrera,Cerro,Alta Cordoba,Alberdi,Jardin,Poeta Lugones
0,nº de medicos,5,12,8,4,15,7
1,Presupuesto (millones de pesos),20,60,35,15,80,25
2,nº de pacientes atendidos por dia,100,350,210,90,450,180
3,nº de cirugias por dia,10,45,25,8,60,20


SelectMultiple(description='Inputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de paci…

SelectMultiple(description='Outputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de pac…

RadioButtons(description='Orientación:', options=('Input', 'Output'), value='Input')

Button(description='Resolver BCC (sin holguras, λ vertical)', style=ButtonStyle())

Output()

In [ ]:
# @title Modelo Aditivo
!pip install pulp openpyxl -q

import pandas as pd
import numpy as np
from pulp import *
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io

def preparar_datos(df_original):
    vars_fila = df_original.iloc[:, 0].values
    dmus = df_original.columns[1:].values
    data = {v: [] for v in vars_fila}
    for dmu in dmus:
        for v in vars_fila:
            val = df_original[df_original.iloc[:, 0] == v][dmu].values[0]
            data[v].append(val)
    df_proc = pd.DataFrame(data, index=dmus)
    df_proc.index.name = 'DMU'
    df_proc.reset_index(inplace=True)
    return df_proc, vars_fila, dmus

def resolver_aditivo_vrs(df, col_dmu, inputs, outputs):
    n = len(df)
    dmu_names = df[col_dmu].values
    X = df[inputs].values.T   # m x n
    Y = df[outputs].values.T  # s x n
    m, s = len(inputs), len(outputs)
    resultados = {
        'ineficiencia': {},   # suma de holguras (sin normalizar)
        'referentes': {},
        'metas_inputs': {},
        'metas_outputs': {}
    }

    for k in range(n):
        prob = LpProblem(f"Aditivo_VRS_{k}", LpMaximize)
        lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
        s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
        s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
        # Objetivo: maximizar suma de holguras (ineficiencia total)
        prob += lpSum(s_menos) + lpSum(s_plus)
        for i in range(m):
            prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == X[i,k]
        for r in range(s):
            prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == Y[r,k]
        # Restricción de convexidad (VRS)
        prob += lpSum(lambdas) == 1
        prob.solve(PULP_CBC_CMD(msg=0))
        lambdas_opt = [value(l) for l in lambdas]
        holg_in = [value(s) for s in s_menos]
        holg_out = [value(s) for s in s_plus]
        suma_holg = sum(holg_in) + sum(holg_out)

        resultados['ineficiencia'][dmu_names[k]] = suma_holg
        resultados['referentes'][dmu_names[k]] = {
            dmu_names[j]: lambdas_opt[j] for j in range(n) if lambdas_opt[j] > 1e-6
        }
        resultados['metas_inputs'][dmu_names[k]] = {inputs[i]: X[i,k] - holg_in[i] for i in range(m)}
        resultados['metas_outputs'][dmu_names[k]] = {outputs[r]: Y[r,k] + holg_out[r] for r in range(s)}
    return resultados

def mostrar_tabla_aditivo(resultados, inputs, outputs, dmus):
    # Filas: θ* (suma holguras), λ*, luego cada input* y cada output*
    rows = ['θ*', 'λ*'] + [f"{inp}*" for inp in inputs] + [f"{out}*" for out in outputs]
    data = {row: [] for row in rows}
    for dmu in dmus:
        inef = resultados['ineficiencia'][dmu]
        data['θ*'].append(f"{inef:.4f}".replace('.', ','))
        # Referentes (solo si la ineficiencia > 0)
        if inef > 1e-6:
            refs = resultados['referentes'][dmu]
            if refs:
                items = sorted(refs.items(), key=lambda x: x[1], reverse=True)
                texto_html = "<br>".join([f"{r} ({w:.2f})".replace('.', ',') for r, w in items])
                data['λ*'].append(texto_html)
            else:
                data['λ*'].append("")
        else:
            data['λ*'].append("")
        # Metas de inputs y outputs
        if inef > 1e-6:
            metas_in = resultados['metas_inputs'][dmu]
            for inp in inputs:
                val = metas_in[inp]
                data[f"{inp}*"].append(f"{val:.2f}".replace('.', ','))
            metas_out = resultados['metas_outputs'][dmu]
            for out in outputs:
                val = metas_out[out]
                data[f"{out}*"].append(f"{val:.2f}".replace('.', ','))
        else:
            for inp in inputs:
                data[f"{inp}*"].append("")
            for out in outputs:
                data[f"{out}*"].append("")
    df_resultado = pd.DataFrame(data, index=dmus).T
    html = df_resultado.to_html(escape=False, na_rep='')
    html = html.replace('<table>', '<td style="text-align:center; vertical-align:middle;">')
    display(HTML(html))

print("📂 Sube archivo Excel (variables en filas, DMUs en columnas)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_orig = pd.read_excel(io.BytesIO(uploaded[filename]))
df_proc, variables, dmus = preparar_datos(df_orig)
print("Vista previa:")
display(df_orig)

inputs_sel = widgets.SelectMultiple(options=list(variables), description='Inputs:')
outputs_sel = widgets.SelectMultiple(options=list(variables), description='Outputs:')
display(inputs_sel, outputs_sel)

btn = widgets.Button(description="Resolver Aditivo (VRS)")
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        try:
            inputs = list(inputs_sel.value)
            outputs = list(outputs_sel.value)
            if not inputs or not outputs:
                print("Selecciona inputs y outputs")
                return
            resultados = resolver_aditivo_vrs(df_proc, 'DMU', inputs, outputs)
            print("\nMODELO ADITIVO (ADD-VRS) - Rendimientos Variables a Escala\n")
            mostrar_tabla_aditivo(resultados, inputs, outputs, dmus)
        except Exception as e:
            print(f"Error: {e}")

btn.on_click(on_click)
display(btn, out)

📂 Sube archivo Excel (variables en filas, DMUs en columnas)


Saving DEA Centros Medicos.xlsx to DEA Centros Medicos (4).xlsx
Vista previa:


,Unnamed: 0,Villa Cabrera,Cerro,Alta Cordoba,Alberdi,Jardin,Poeta Lugones
0,nº de medicos,5,12,8,4,15,7
1,Presupuesto (millones de pesos),20,60,35,15,80,25
2,nº de pacientes atendidos por dia,100,350,210,90,450,180
3,nº de cirugias por dia,10,45,25,8,60,20


SelectMultiple(description='Inputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de paci…

SelectMultiple(description='Outputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de pac…

Button(description='Resolver Aditivo (VRS)', style=ButtonStyle())

Output()

In [27]:
# @title Eficiencia de Escala (CCR + BCC)
!pip install pulp openpyxl -q

import pandas as pd
import numpy as np
from pulp import *
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io

def preparar_datos(df_original):
    vars_fila = df_original.iloc[:, 0].values
    dmus = df_original.columns[1:].values
    data = {v: [] for v in vars_fila}
    for dmu in dmus:
        for v in vars_fila:
            val = df_original[df_original.iloc[:, 0] == v][dmu].values[0]
            data[v].append(val)
    df_proc = pd.DataFrame(data, index=dmus)
    df_proc.index.name = 'DMU'
    df_proc.reset_index(inplace=True)
    return df_proc, vars_fila, dmus

def resolver_ccr(df, col_dmu, inputs, outputs, orientacion):
    n = len(df)
    dmu_names = df[col_dmu].values
    X = df[inputs].values.T
    Y = df[outputs].values.T
    m, s = len(inputs), len(outputs)
    resultados = {'eficiencia': {}, 'referentes': {}}
    EPS = 1e-6
    for k in range(n):
        if orientacion == "Input":
            prob = LpProblem(f"CCR_Input_{k}", LpMinimize)
            theta = LpVariable("theta", lowBound=0, upBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            prob += theta + EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == theta * X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == Y[r,k]
            prob.solve(PULP_CBC_CMD(msg=0))
            theta_opt = value(theta)
            eficiencia = theta_opt
            lambdas_opt = [value(l) for l in lambdas]
        else:  # Output
            prob = LpProblem(f"CCR_Output_{k}", LpMaximize)
            phi = LpVariable("phi", lowBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            prob += phi - EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == phi * Y[r,k]
            prob.solve(PULP_CBC_CMD(msg=0))
            phi_opt = value(phi)
            eficiencia = 1.0 / phi_opt if phi_opt > 0 else 0
            lambdas_opt = [value(l) for l in lambdas]
        resultados['eficiencia'][dmu_names[k]] = eficiencia
        resultados['referentes'][dmu_names[k]] = {dmu_names[j]: lambdas_opt[j] for j in range(n) if lambdas_opt[j] > 1e-6}
    return resultados

def resolver_bcc(df, col_dmu, inputs, outputs, orientacion):
    n = len(df)
    dmu_names = df[col_dmu].values
    X = df[inputs].values.T
    Y = df[outputs].values.T
    m, s = len(inputs), len(outputs)
    resultados = {'eficiencia': {}, 'referentes': {}}
    EPS = 1e-6
    for k in range(n):
        if orientacion == "Input":
            prob = LpProblem(f"BCC_Input_{k}", LpMinimize)
            theta = LpVariable("theta", lowBound=0, upBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            prob += theta + EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == theta * X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == Y[r,k]
            prob += lpSum(lambdas) == 1
            prob.solve(PULP_CBC_CMD(msg=0))
            theta_opt = value(theta)
            eficiencia = theta_opt
            lambdas_opt = [value(l) for l in lambdas]
        else:  # Output
            prob = LpProblem(f"BCC_Output_{k}", LpMaximize)
            phi = LpVariable("phi", lowBound=1)
            lambdas = [LpVariable(f"l_{j}", lowBound=0) for j in range(n)]
            s_menos = [LpVariable(f"s_in_{i}", lowBound=0) for i in range(m)]
            s_plus = [LpVariable(f"s_out_{r}", lowBound=0) for r in range(s)]
            prob += phi - EPS * (lpSum(s_menos) + lpSum(s_plus))
            for i in range(m):
                prob += lpSum(lambdas[j]*X[i,j] for j in range(n)) + s_menos[i] == X[i,k]
            for r in range(s):
                prob += lpSum(lambdas[j]*Y[r,j] for j in range(n)) - s_plus[r] == phi * Y[r,k]
            prob += lpSum(lambdas) == 1
            prob.solve(PULP_CBC_CMD(msg=0))
            phi_opt = value(phi)
            eficiencia = 1.0 / phi_opt if phi_opt > 0 else 0
            lambdas_opt = [value(l) for l in lambdas]
        resultados['eficiencia'][dmu_names[k]] = eficiencia
        resultados['referentes'][dmu_names[k]] = {dmu_names[j]: lambdas_opt[j] for j in range(n) if lambdas_opt[j] > 1e-6}
    return resultados

def mostrar_tabla_es(dmus, eff_ccr, refs_ccr, eff_bcc, refs_bcc, orientacion):
    # Calcular ES = eficiencia_CCR / eficiencia_BCC (para ambas orientaciones)
    es_values = []
    for dmu in dmus:
        ccr = eff_ccr[dmu]
        bcc = eff_bcc[dmu]
        if ccr is None or bcc is None or bcc == 0:
            es = 0.0
        else:
            es = ccr / bcc
            if es > 1 + 1e-6:
                es = 1.0
            elif es > 1.0:
                es = 1.0
        es_values.append(es)

    filas = {
        'CCR (θ*)': [],
        'λ* CCR': [],
        'BCC (θ*)': [],
        'λ* BCC': [],
        'ES (Escala)': []
    }
    for i, dmu in enumerate(dmus):
        filas['CCR (θ*)'].append(f"{eff_ccr[dmu]:.4f}".replace('.', ',') if eff_ccr[dmu] is not None else "")
        # λ* CCR
        refs_c = refs_ccr[dmu]
        if refs_c and eff_ccr[dmu] < 0.9999:
            items = sorted(refs_c.items(), key=lambda x: x[1], reverse=True)
            texto = "<br>".join([f"{r} ({w:.2f})".replace('.', ',') for r, w in items])
            filas['λ* CCR'].append(texto)
        else:
            filas['λ* CCR'].append("")
        # BCC θ*
        filas['BCC (θ*)'].append(f"{eff_bcc[dmu]:.4f}".replace('.', ',') if eff_bcc[dmu] is not None else "")
        # λ* BCC
        refs_b = refs_bcc[dmu]
        if refs_b and eff_bcc[dmu] < 0.9999:
            items = sorted(refs_b.items(), key=lambda x: x[1], reverse=True)
            texto = "<br>".join([f"{r} ({w:.2f})".replace('.', ',') for r, w in items])
            filas['λ* BCC'].append(texto)
        else:
            filas['λ* BCC'].append("")
        # ES
        filas['ES (Escala)'].append(f"{es_values[i]:.4f}".replace('.', ','))

    df_resultado = pd.DataFrame(filas, index=dmus).T
    html = df_resultado.to_html(escape=False, na_rep='')
    html = html.replace('<tr>', '<td style="text-align:center; vertical-align:middle;">')
    display(HTML(html))

print("📂 Sube archivo Excel (variables en filas, DMUs en columnas)")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_orig = pd.read_excel(io.BytesIO(uploaded[filename]))
df_proc, variables, dmus = preparar_datos(df_orig)
print("Vista previa:")
display(df_orig)

inputs_sel = widgets.SelectMultiple(options=list(variables), description='Inputs:')
outputs_sel = widgets.SelectMultiple(options=list(variables), description='Outputs:')
orientacion_sel = widgets.RadioButtons(options=['Input', 'Output'], description='Orientación:', value='Input')
display(inputs_sel, outputs_sel, orientacion_sel)

btn = widgets.Button(description="Calcular Eficiencia de Escala")
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        try:
            inputs = list(inputs_sel.value)
            outputs = list(outputs_sel.value)
            orientacion = orientacion_sel.value
            if not inputs or not outputs:
                print("Selecciona inputs y outputs")
                return
            resultados_ccr = resolver_ccr(df_proc, 'DMU', inputs, outputs, orientacion)
            resultados_bcc = resolver_bcc(df_proc, 'DMU', inputs, outputs, orientacion)
            eff_ccr = resultados_ccr['eficiencia']
            refs_ccr = resultados_ccr['referentes']
            eff_bcc = resultados_bcc['eficiencia']
            refs_bcc = resultados_bcc['referentes']
            print(f"\nEFICIENCIA DE ESCALA - Orientación {orientacion}\n")
            mostrar_tabla_es(dmus, eff_ccr, refs_ccr, eff_bcc, refs_bcc, orientacion)
        except Exception as e:
            print(f"Error: {e}")

btn.on_click(on_click)
display(btn, out)

📂 Sube archivo Excel (variables en filas, DMUs en columnas)


Saving DEA Centros Medicos.xlsx to DEA Centros Medicos (10).xlsx
Vista previa:


,Unnamed: 0,Villa Cabrera,Cerro,Alta Cordoba,Alberdi,Jardin,Poeta Lugones
0,nº de medicos,5,12,8,4,15,7
1,Presupuesto (millones de pesos),20,60,35,15,80,25
2,nº de pacientes atendidos por dia,100,350,210,90,450,180
3,nº de cirugias por dia,10,45,25,8,60,20


SelectMultiple(description='Inputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de paci…

SelectMultiple(description='Outputs:', options=('nº de medicos', 'Presupuesto (millones de pesos)', 'nº de pac…

RadioButtons(description='Orientación:', options=('Input', 'Output'), value='Input')

Button(description='Calcular Eficiencia de Escala', style=ButtonStyle())

Output()